# 00 · Data preparation  (CPU · ~0.5–1 h · no GPU quota)

Locates and verifies all datasets, indexes CASIA-WebFace identity labels, and builds the aligned LFW
identification gallery. Output (`/kaggle/working/data`) is the input of every later notebook.

**Notebook settings:** Accelerator *None*, Internet *On*.

**Attach these Kaggle datasets (Add Input):**
1. **CASIA-WebFace in InsightFace format** — search *"faces_webface_112x112"* / *"casia webface insightface"*.
   Must contain `train.rec`, `train.idx`, and the benchmark bins `lfw.bin`, `cfp_fp.bin`, `agedb_30.bin`.
2. *(optional)* a dataset with `calfw.bin` and `cplfw.bin` (e.g. the *faces_emore* / *ms1m-retinaface* packages).
3. **`jessicali9530/lfw-dataset`** — LFW images for the identification gallery / demo.

After it finishes: **Save Version → Save & Run All**, then attach this notebook's output to notebook 01.

In [ ]:
# Setup: clone the repo (code lives in src/fpgaface, notebooks are thin wrappers)
import os
REPO = "https://github.com/sidwastaken07/face-detection.git"
BRANCH = "claude/keen-meitner-ab0d8k"
CODE = "/tmp/face-detection"
if not os.path.exists(CODE):
    !git clone -q -b {BRANCH} {REPO} {CODE}
else:
    !git -C {CODE} pull -q
%cd {CODE}
!pip install -q -e . onnx onnx2torch
!pip install -q --no-deps facenet-pytorch
# A running kernel does not see a fresh editable install; make the package importable directly.
import sys
if f"{CODE}/src" not in sys.path:
    sys.path.insert(0, f"{CODE}/src")


In [ ]:
!find /kaggle/input -maxdepth 3 | head -50

In [ ]:
# Indexes CASIA labels (~1–2 min), checks every benchmark, MTCNN-aligns the LFW gallery (~10–15 min on CPU)
!python scripts/prepare_data.py --out /kaggle/working/data --device cpu

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from fpgaface.utils import load_manifest
m = load_manifest("/kaggle/working/data")
print(json.dumps(m, indent=2))

labels = np.load(m["casia"]["labels"])
counts = np.bincount(labels)
plt.figure(figsize=(6, 3)); plt.hist(counts[counts > 0], bins=100, log=True)
plt.xlabel("images per identity"); plt.ylabel("identities"); plt.title("CASIA-WebFace"); plt.show()

In [ ]:
# Visual check of the aligned gallery: aligned 112×112 (top) vs FaceNet 160×160 crop (bottom)
g = np.load(m["lfw_gallery"])
print("images:", len(g["labels"]), "identities:", len(g["persons"]), "MTCNN detected:", g["detected"].mean())
idx = np.random.default_rng(0).choice(len(g["labels"]), 8, replace=False)
fig, ax = plt.subplots(2, 8, figsize=(16, 4))
for j, i in enumerate(idx):
    ax[0, j].imshow(g["aligned112"][i]); ax[1, j].imshow(g["facenet160"][i])
    ax[0, j].set_title(str(g["persons"][g["labels"][i]])[:14], fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.show()